# 01 · Data preparation — §1

Draws the one split every model uses. **Run it in official mode once, early** — until its three files are committed, the other notebooks can only run in `synthetic` mode. In `synthetic` / `debug` mode it prepares and shows everything but writes nothing.

| | |
|---|---|
| **Owner** | Member 1 |
| **Reads** | EuroSAT via torchvision → `data/raw/` |
| **Writes (official)** | `data/splits/split_manifest.csv`, `split_meta.json`, `norm_stats.json`, `results/figures/dataset/*.png` |
| **Report needs** | dataset description + citation, split table, class distribution, sample images |

In [ ]:
# ── Cell 1 · Colab bootstrap ─ does nothing when run locally ───────────────
# Locally the package is already installed (pip install -e .), so this is skipped.
# On Colab it clones the repo, installs it and makes it importable.
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import os
    import subprocess

    BRANCH = "main"  # <- your branch, to run work that is not merged yet
    REPO_URL = "https://github.com/ThejithaR/EN3150-Assignment-03-CNN.git"
    REPO_DIR = "/content/EN3150-Assignment-03-CNN"

    if not os.path.exists(REPO_DIR):
        env = dict(os.environ)
        try:  # only needed if the repo is private; the token never leaves the environment
            from google.colab import userdata

            token = userdata.get("GH_TOKEN")
            if token:
                env["GH_TOKEN"] = token
        except Exception:
            pass
        helper = "!f() { echo username=x-access-token; echo password=$GH_TOKEN; }; f"
        subprocess.run(
            ["git", "-c", f"credential.helper={helper}", "clone", "--quiet",
             "--branch", BRANCH, REPO_URL, REPO_DIR],
            env=env,
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO_DIR], check=True)
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")  # an editable install is only seen after a restart
    os.chdir(REPO_DIR)
    print(f"Colab ready: {REPO_DIR} @ {BRANCH}")

In [ ]:
# ── Cell 2 · Setup ─────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2

# MODE decides how much runs and whether anything is written:
#   "synthetic"  generated data, 2 epochs, seconds         - before EuroSAT exists
#   "debug"      5% of EuroSAT, 2 epochs                   - find bugs step by step
#   "official"   the full run - the ONLY mode that writes  - then Restart & Run All
MODE = "synthetic"

from edgecnn.config import load_config
from edgecnn.contracts import paths, schema
from edgecnn.data import plot_class_distribution, plot_sample_grid, prepare_dataset, split_counts_table

In [ ]:
# Shared figure style (Member 1): small inline figures, print-quality saved PNGs.
from edgecnn.evaluation.plotting import apply_style

apply_style()

## Config

Data settings are shared by every experiment, so any experiment file works here.

In [ ]:
cfg = load_config("configs/experiments/model_b__adam.yaml", mode=MODE)
print(f"mode={cfg.mode}  official={cfg.is_official}")
print("dataset:", cfg.section("dataset")["name"], "| split:", cfg.section("split"))

## 1 · Prepare the dataset

Downloads EuroSAT, draws the stratified 70/15/15 split and computes train-only normalisation. An existing committed split is never overwritten (`force=False`).

In [ ]:
prepared = prepare_dataset(cfg)

## 2 · Split sizes

Check the totals row shows 70 / 15 / 15, and no class has zero images in any split.

In [ ]:
split_counts_table(prepared["meta"])

## 3 · Class distribution

The bars for each class should have the same proportions in every split.

In [ ]:
plot_class_distribution(prepared["meta"], write=cfg.is_official)

## 4 · Sample images

What the models actually see: 64×64, no augmentation.

In [ ]:
plot_sample_grid(cfg, per_class=4, write=cfg.is_official)

## 5 · Normalisation statistics

`fitted_on` must be `train`.

In [ ]:
prepared["norm_stats"]

## 6 · Validate the committed files

Only meaningful after an official run has written them.

In [ ]:
if paths.SPLIT_MANIFEST.exists():
    rows = schema.validate_manifest(paths.SPLIT_MANIFEST)
    meta = schema.read_json(paths.SPLIT_META, "split_meta")
    schema.read_json(paths.NORM_STATS, "norm_stats")
    print(f"valid: {rows} manifest rows, {meta['num_classes']} classes, counts {meta['counts']}")
else:
    print("No committed split yet - run this notebook with MODE = 'official'.")

## Official-run checklist

- [ ] `MODE = "official"` in the setup cell
- [ ] Kernel → **Restart & Run All** finished without errors
- [ ] every output is what you expect — no stray warnings or giant prints
- [ ] announce in the group chat that the split is committed — everyone can leave `synthetic`
- [ ] `pytest tests/contracts` is green
- [ ] commit this notebook **with its outputs**, together with the files it wrote

In [ ]:
# ── Colab only: send this run's results back to GitHub ─────────────────────
# Does nothing locally, or in synthetic / debug mode. Procedure: notebooks/README.md
if IN_COLAB and MODE == "official":
    from edgecnn.utils import push_results

    push_results(
        [paths.SPLITS_DIR, paths.DATASET_FIGURES_DIR],
        "Add EuroSAT split (Colab)",
    )